<a href="https://colab.research.google.com/github/sathushetty7/RAG-PDF-Chatbot/blob/main/PDF_RAG_Question_Answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q pypdf langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [13]:
import os
import torch

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cpu
CUDA available: False


In [14]:
from google.colab import files

uploaded = files.upload()

Saving pdf-Attention Is All You Need.pdf to pdf-Attention Is All You Need.pdf


In [15]:
pdf_path = next(iter(uploaded))

print("PDF path:", pdf_path)

PDF path: pdf-Attention Is All You Need.pdf


In [16]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

print("Number of pages:", len(reader.pages))
print("Characters extracted:", len(text))

print("\n--- FIRST 2000 CHARACTERS ---\n")
print(text[:2000])

Number of pages: 15
Characters extracted: 39511

--- FIRST 2000 CHARACTERS ---

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on a

In [51]:
# SPLIT PDF TEXT INTO CHUNKS

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create smaller overlapping chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

# Split PDF text into chunks
chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))
print("\n--- FIRST CHUNK ---\n")
print(chunks[0])

Number of chunks: 60

--- FIRST CHUNK ---

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best


In [18]:
# Load the embedding model
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert chunks into embeddings
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (60, 384)


In [19]:
#  BUILD FAISS VECTOR INDEX

import faiss
import numpy as np

# Convert embeddings to float32
embedding_matrix = np.array(
    embeddings
).astype("float32")

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embedding_matrix)

# Create cosine similarity index
index = faiss.IndexFlatIP(
    embedding_matrix.shape[1]
)

# Add embeddings to the index
index.add(embedding_matrix)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 60


In [20]:
# LOAD FLAN-T5 MODEL

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

# Load FLAN-T5 model
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

print("FLAN-T5 loaded successfully.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 loaded successfully.


In [54]:
#10 INSPECT RETRIEVED CHUNKS

question = input("Ask a question about the PDF: ")

# Convert question into an embedding
query_embedding = embedding_model.encode(
    [question]
).astype("float32")

# Normalize for cosine similarity
faiss.normalize_L2(query_embedding)

# Retrieve the top 5 chunks
similarities, indices = index.search(
    query_embedding,
    5
)

print("\n__________RETRIEVED CHUNKS_____________")

for i, idx in enumerate(indices[0]):
    print(
        f"\n--- Chunk {i + 1} | "
        f"Similarity: {similarities[0][i]:.4f} ---"
    )
    print(chunks[idx])

Ask a question about the PDF: What is multi-head attention?

__________RETRIEVED CHUNKS_____________

--- Chunk 1 | Similarity: 0.6209 ---
linear projections todk,dk anddv dimensions, respectively. On each of these projected versions of
queries, keys and values we then perform the attention function in parallel, yieldingdv-dimensional
4To illustrate why the dot products get large, assume that the components of q and k are independent random
variables with mean 0 and variance 1. Then their dot product, q · k = Pdk
i=1 qiki, has mean 0 and variance dk.
4
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O

--- Chunk 2 | Similarity: 0.6012 ---
sub-layer in the decoder stack to prevent po

In [63]:
# QUESTION ANSWERING WITH RAG

def ask_question(query, top_k=5):

    # Convert question into an embedding
    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Retrieve relevant chunks
    similarities, indices = index.search(
        query_embedding,
        top_k
    )

    # Use only the most relevant chunk
    context = chunks[indices[0][0]]

    # Create focused prompt
    prompt = f"""
Read the context and answer the question.

Context:
{context}

Question:
{query}

Write one complete sentence that directly answers the question.
Start the sentence with the subject or concept being asked about.
Do not begin with a fragment.
Use only information from the context.

Answer:
"""

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    # Generate answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    # Decode answer
    result_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return result_text.strip()


In [64]:
question = input("Ask a question about the PDF: ")

answer = ask_question(question)

print("\n__________ANSWER_____________")
print(answer)

Ask a question about the PDF:  What is multi-head attention?

__________ANSWER_____________
Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions.
